In [ ]:
# CNN1

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data():
    """
    Loads (140,4) data from "Feature_CNN1.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 140, 4)
    return X_branch1

def load_reaction_rates():
    with open('HT1-1_indel_frequency_value_percentage.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)

########################################
# 2. Branch
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (140,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*70, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,70)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,70)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x

########################################
# 3. Single-branch Model (CNN1-only)
########################################

class CNN1_Only(nn.Module):
    """
    End-to-end: 
      - CNNBranch1 => feat_cnn1
    dropout => final FC => 1
    """
    def __init__(self,
                 filters1, kernel_size1, dense_units1,  # CNN1
                 final_fc_dim,
                 dropout_rate=0.0):  # new hyperparameter for dropout
        super().__init__()
        
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        
        # total dimension = (dense_units1)
        total_dim = dense_units1
        
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(total_dim, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat_cnn1 = self.cnn_branch1(x1)                   # (batch, dense_units1)

        feat_cnn1 = F.dropout(feat_cnn1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(feat_cnn1))   # (batch, final_fc_dim)
        out = self.out(x)                 # (batch, 1)
        return out.view(-1)

########################################
# 4. Dataset
########################################

class IndivDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
        
        assert len(X1) == self.num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)  
        y = torch.tensor(self.reaction_rates[idx], dtype=torch.float)
        return x1, y

def collate(batch):
    from torch_geometric.data import Batch
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y

########################################
# 5. Objective Function
########################################

import copy

# Global tracking across trials
best_trial_val_loss = float("inf")
best_model_global = None

def objective(trial):
    # CNN1
    filters1 = trial.suggest_int("filters1", 16, 128, step=16)
    kernel_size1 = trial.suggest_categorical("kernel_size1", [3,5,7])
    dense_units1 = trial.suggest_int("dense_units1", 16, 256, step=16)

    # final aggregator
    final_fc_dim = trial.suggest_int("final_fc_dim", 16, 256, step=16)

    # Dropout hyperparameter (0.2 to 0.7)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    
    # Learning rate and batch size
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [8,16,32])
    epochs = 100
    patience = 10
    
    loss_fn = nn.MSELoss()
    
    # load data
    X1 = load_branch1_data()
    rates = load_reaction_rates()
    
    indiv_dataset = IndivDataset(X1, rates)
    
    # Splitting the dataset
    np.random.seed(42)
    full_indices = np.arange(len(rates))
    selected_indices = np.random.choice(len(full_indices), size=12000, replace=False)
    unseen_indices = np.setdiff1d(full_indices, selected_indices)
    train_idx, val_idx = train_test_split(selected_indices, test_size=0.2, random_state=42)
    
    train_set = Subset(indiv_dataset, train_idx)
    val_set = Subset(indiv_dataset, val_idx)
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, collate_fn=collate)
    
    # Build the model
    model = CNN1_Only(
        filters1, kernel_size1, dense_units1,
        final_fc_dim,
        dropout_rate=dropout_rate
    )
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    best_val_loss = float('inf')
    counter = 0
    best_model_state = None  # To store the best model's state
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        total_loss = 0.0
        for x1_b, y_b in train_loader:
            pred = model(x1_b)
            loss = loss_fn(pred, y_b.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_train_loss = total_loss / len(train_loader)

        # Validation phase
        model.eval()
        total_val = 0.0
        with torch.no_grad():
            for x1_b, y_b in val_loader:
                valpred = model(x1_b)
                vloss = loss_fn(valpred, y_b.view(-1))
                total_val += vloss.item()
        avg_val_loss = total_val / len(val_loader)
        
        trial.report(avg_val_loss, epoch)
        
        # Check for improvement
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model)
            counter = 0
        else:
            counter += 1

        # Early stopping if no improvement for 'patience' epochs
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
        
        print(f"Epoch {epoch+1}/{epochs} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")
    
    # Load best model from this trial
    if best_model_state is not None:
        model = best_model_state
    
    # Save only if this trial is the best across ALL trials
    global best_trial_val_loss
    if best_val_loss < best_trial_val_loss:
        best_trial_val_loss = best_val_loss
        torch.save(model, "best_CNN1.pt")
        print(f"New best model saved with val_loss = {best_val_loss:.6f}")

    unseen_set = Subset(indiv_dataset, unseen_indices)
    model.eval()
    trial_loader = DataLoader(unseen_set, batch_size=1, shuffle=False, collate_fn=collate)
    preds_trial = []
    labels_trial = []
    with torch.no_grad():
        for x1_b, y_b in trial_loader:
            p = model(x1_b)
            preds_trial.append(p.item())
            labels_trial.append(y_b.item())
    sp_corr, _ = spearmanr(labels_trial, preds_trial)
    print(f"Spearman for best model = {sp_corr}")
    
    return best_val_loss

########################################
# 6. Main Pipeline
########################################

def main_pipeline():
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=10)
    
    print("Best hyperparameters:", study.best_trial.params)
    best_trial_number = study.best_trial.number
    best_params = study.best_trial.params
    
    # Rebuild for final 100-trial evaluation by loading the entire saved model
    model = torch.load(f"best_CNN1.pt", weights_only=False)
    model.eval()
    
    # final hold-out evaluation
    X1 = load_branch1_data()
    y = load_reaction_rates()
    
    indiv_dataset = IndivDataset(X1, y)
    
    np.random.seed(42)
    full_indices = np.arange(len(y))
    selected_indices = np.random.choice(len(full_indices), size=12000, replace=False)
    unseen_indices = np.setdiff1d(full_indices, selected_indices)
    unseen_set = Subset(indiv_dataset, unseen_indices)
    
    trial_loader = DataLoader(unseen_set, batch_size=1, shuffle=False, collate_fn=collate)
    preds_trial = []
    labels_trial = []
    with torch.no_grad():
        for x1_b, y_b in trial_loader:
            p = model(x1_b)
            preds_trial.append(p.item())
            labels_trial.append(y_b.item())
    sp_corr, _ = spearmanr(labels_trial, preds_trial)
    print(f"Spearman for best model = {sp_corr}")

if __name__ == "__main__":
    main_pipeline()